In [ ]:
'''
    Code to compute p-values of variable interactions with logistic regression
    Supplemental Figures S6
'''

import pandas as pd
import numpy as np
import random
import time

from options.options import Options
import util.util as util
import util.pre_process as pre
import util.bootstrap_tools as btool

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import numpy as np

import torch
from scipy.stats import norm

MIN_VAL = 20 # minimum number of cases in strata for analysis, otherwise skip

phen_label = 'MDD' # set the phenotype to be examined

opt = Options()
opt.initialize()

# Determine eMERGE cutoff points
emerge_cutoff = opt.phen_params[phen_label]['emerge_cut']
pgs_emerge_cutoff = norm.ppf(1-emerge_cutoff)

# load PGS and covar files
pgs_df = pd.read_csv(opt.prs_file, delim_whitespace=True, index_col='IID')
covars_df = pd.read_csv(opt.env_file, sep='\t', index_col='IID')

pc_cols = ['chip']+[df_col for df_col in covars_df.columns if df_col.startswith('PC')]
pcs_df = covars_df[pc_cols].iloc[:, :11]
covars_df = covars_df[opt.covars]

# Get indices of subpopulations
phen = pgs_df[opt.phen_params[phen_label]['phen']]
phen = phen.dropna()
pgs = pgs_df[opt.phen_params[phen_label]['pgs']].loc[phen.index]
pgs = (pgs-pgs.mean())/pgs.std()
if opt.phen_params[phen_label]['pgs_flip']:
    pgs = -1*pgs
covars_df = covars_df.loc[phen.index]

bin_defs = opt.bin_defs
bounds, labels, label_map = util.create_bins_and_indices(covars_df, bin_defs)

# impute values in matrix
covars_df = pre.impute_covars(covars_df)
covars_df, _, _ = pre.standardize_covars(covars_df)
covars_df = covars_df.loc[pgs.index]

pcs_df = pcs_df.loc[pgs.index]

util.create_folder(f'results/{phen_label}')

In [ ]:
def significant_diff(model, i, j):
    w = model.params
    se = model.bse
    diff = w[i] - w[j]
    se_ij = (se[i]**2 + se[j]**2)**0.5
    from scipy.stats import norm
    z = diff / se_ij
    p = 2 * (1 - norm.cdf(abs(z)))
    return p

In [ ]:
import statsmodels.api as sm 

model_high_dict = {}
model_full_dict = {}
label_lookup = {c: [k for k, v in enumerate(labels) if c in v] for c in opt.covars}

for i in range(len(opt.covars)):
    ci = opt.covars[i]
    print(ci) 
    
    ci_idx = label_lookup[ci]

    ni = len(ci_idx)
    n_strata = ni
    factors = pd.DataFrame(0.0, index=pgs.index, columns=range(n_strata))

    for a, ii in enumerate(ci_idx):
        idx = label_map[labels[ii]]
        factors.loc[idx, a] = 1.0

    X = np.column_stack([
            factors.mul(pgs, axis=0).values,
            factors.values,
            covars_df.values,
            pcs_df.values,
            np.ones((len(pgs), 1)),
        ])
    model_full = sm.Logit(phen.values, X).fit(disp=0)

    X[:,:n_strata] = factors.mul(pgs > pgs_emerge_cutoff, axis=0).values
    model_high = sm.Logit(phen.values, X).fit(disp=0)

    key = f"{ci}"
    model_high_dict[key] = model_high
    model_full_dict[key] = model_full

    idx_max = model_full.params[:n_strata].argmax()
    idx_min = model_full.params[:n_strata].argmin()
    print("FULL", significant_diff(model_full, idx_min, idx_max)) # i.e. using full PGS distribution

    idx_max = model_high.params[:n_strata].argmax()
    idx_min = model_high.params[:n_strata].argmin()
    print("HIGH", significant_diff(model_high, idx_min, idx_max)) #i.e. binarizing PGS distribution
    
    for j in range(i + 1, len(opt.covars)):
        cj = opt.covars[j]
        cj_idx = label_lookup[cj]
        print(ci, cj)

        ni = len(ci_idx)
        nj = len(cj_idx)
        n_strata = ni * nj

        factors = pd.DataFrame(0.0, index=pgs.index, columns=range(n_strata))

        for a, ii in enumerate(ci_idx):
            for b, jj in enumerate(cj_idx):
                k = a * nj + b
                idx = np.intersect1d(label_map[labels[ii]], label_map[labels[jj]])
                factors.loc[idx, k] = 1.0

        X = np.column_stack([
            factors.mul(pgs, axis=0).values,
            factors.values,
            covars_df.values,
            pcs_df.values,
            np.ones((len(pgs), 1)),
        ])
        model_full = sm.Logit(phen.values, X).fit(disp=0)

        X[:,:n_strata] = factors.mul(pgs > pgs_emerge_cutoff, axis=0).values
        model_high = sm.Logit(phen.values, X).fit(disp=0)

        key = f"{ci}_{cj}"
        model_high_dict[key] = model_high
        model_full_dict[key] = model_full

        idx_max = model_full.params[:n_strata].argmax()
        idx_min = model_full.params[:n_strata].argmin()
        print("FULL", significant_diff(model_full, idx_min, idx_max))

        idx_max = model_high.params[:n_strata].argmax()
        idx_min = model_high.params[:n_strata].argmin()
        print("HIGH", significant_diff(model_high, idx_min, idx_max))

np.save(f"results/{phen_label}/model_full.npy", model_full_dict)
np.save(f"results/{phen_label}/model_high.npy", model_high_dict)